### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos el archivo movie_cast.json de nuestro contenedor bronze

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
movie_cast_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
])

In [0]:
df = spark.read \
    .schema(movie_cast_schema) \
    .option("multiline", True) \
    .json(f"{bronze_folder_path}/movie_cast.json")
    

###### Eliminamos las columnas que no nos interesan

In [0]:
from pyspark.sql.functions import col,current_timestamp, lit
df_drop = df.drop(col("genderId"))\
            .drop(col("castOrder"))

###### Cambiamos el nombre de las columnas

In [0]:
df_renamed = df_drop.withColumnRenamed("movieId", "movie_id")\
                    .withColumnRenamed("personId", "person_id")\
                    .withColumnRenamed("characterName", "character_name")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
df_add = add_columnas_control(df_renamed,v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_cast")